In [1]:
import time

from dotenv import load_dotenv
from loguru import logger

from data_processing.temporal_kg_engine.factory import TKGFactory
from data_processing.temporal_kg_engine.in_memory import InMemoryTKGEngine

load_dotenv()

True

In [2]:
engine_name = "in_memory"
# Create engine from environment
engine: InMemoryTKGEngine = TKGFactory.from_env(engine_name)

logger.debug(engine)

2025-11-07 12:37:41.989 | DEBUG    | data_processing.temporal_kg_engine.factory:from_env:34 - GRAPH_NAME='pubtator3-v0.4.0' BATCH_SIZE=100000 NODE_PROPERTIES={'node_id': 'node_id', 'node_name': 'node_name'} LOCAL_DATA_STORAGE_DIR='/home/pablo.sanchez2/data/{graph_name}' S3_BACKUP_DIR='s3://sai-spaice-ds/data/processed/data_processing/{graph_name}/backups' S3_FACTS_TSV_PATH='s3://sai-spaice-ds/data/processed/data_processing/{graph_name}/edges.tsv' S3_NODES_TSV_PATH='s3://sai-spaice-ds/data/processed/data_processing/{graph_name}/nodes_corrected.tsv'
2025-11-07 12:37:41.990 | INFO     | data_processing.temporal_kg_engine.in_memory.engine:__init__:43 - Initialized InMemoryTKGEngine for graph: pubtator3-v0.4.0
2025-11-07 12:37:41.990 | DEBUG    | __main__:<module>:5 - InMemoryTKGEngine(graph_name='pubtator3-v0.4.0', batch_size=100000, nodes=0, edges=0)


In [3]:
engine.load_data()

2025-11-07 12:37:43.132 | INFO     | data_processing.temporal_kg_engine.base:load_data:260 - ============================================================
2025-11-07 12:37:43.133 | INFO     | data_processing.temporal_kg_engine.base:load_data:261 - STEP 1: Loading Data
2025-11-07 12:37:43.133 | INFO     | data_processing.temporal_kg_engine.base:load_data:262 - ============================================================
2025-11-07 12:37:43.191 | INFO     | data_processing.temporal_kg_engine.base:load_data:279 - ✅ Loaded 580,434 nodes from /home/pablo.sanchez2/data/pubtator3-v0.4.0/nodes_corrected.tsv
2025-11-07 12:37:43.531 | INFO     | data_processing.temporal_kg_engine.base:load_data:295 - ✅ Loaded 11,508,914 edges from /home/pablo.sanchez2/data/pubtator3-v0.4.0/edges.tsv
2025-11-07 12:37:43.558 | INFO     | data_processing.temporal_kg_engine.base:load_data:305 - ✅ 11,508,914 edges ready for ingestion
2025-11-07 12:37:43.573 | INFO     | data_processing.temporal_kg_engine.base:load_dat

In [4]:
excluded_node = "be61640182934a2527000895202e3a2e"

sql = f"""
SELECT DISTINCT subject_id AS node_id
FROM edges
WHERE subject_domain = 'DISEASE'
  AND NOT (relation_type = 'ASSOCIATE' AND object_id = '{excluded_node}')
"""
print(sql)

sql = """
query = "MATCH (s)-[r]->(o) WHERE (('DISEASE' IN LABELS(o)) AND (NOT (s.node_id = 'e7c783922a15c21e49ddee3a4b4b3129' AND TYPE(r) = 'ASSOCIATE' AND 'DISEASE' IN LABELS(o)))) RETURN DISTINCT(o.node_id) AS node_id"
"""


excluded_node = "e7c783922a15c21e49ddee3a4b4b3129"

sql = f"""
SELECT DISTINCT object_id AS node_id
FROM edges
WHERE object_domain = 'DISEASE'
  AND NOT(relation_type = 'ASSOCIATE' AND subject_id = '{excluded_node}')
"""

tic = time.perf_counter()
results = engine.sql_query(sql, tables=["edges"])
delay = time.perf_counter() - tic
print(f"[{delay:.2f} secs] Num results: {len(results)}")

for row in results[:10]:
    object_node_id = row["node_id"]
    sql_2 = f"""
    SELECT subject_id, relation_type, object_id
    FROM edges
    WHERE object_domain = 'DISEASE' AND relation_type = 'ASSOCIATE' AND subject_id = '{excluded_node}' AND object_id = '{object_node_id}'
    """
    results_2 = engine.sql_query(sql_2, tables=["edges"])

    print(len(results_2))
    print(object_node_id)


SELECT DISTINCT subject_id AS node_id
FROM edges
WHERE subject_domain = 'DISEASE'
  AND NOT (relation_type = 'ASSOCIATE' AND object_id = 'be61640182934a2527000895202e3a2e')

[0.10 secs] Num results: 9969
1
38720b76d83a57a04d5b763dc0863436
1
3893283264cd9e3c32793cf194f2efa2
1
fefeaf7a1e2e4a24e3600ec57cfe945a
1
4dd98c9520e2bd63a7bbf614cd101f6d
1
35fe8a9f3f8b5d1dd88fe62501746e8f
1
9ef2c3da5f374f92db814ca7de5d609a
1
a390c19f209b18f883fb3e87a7953586
1
c13564ae13479c405f071d96c4baf4e8
1
87c8fa06f9051f3038b37147dc41d8fa
1
de7e32f56378779c862f5edc09a237bb


In [5]:
engine.connect()

2025-11-07 12:37:45.689 | INFO     | data_processing.temporal_kg_engine.in_memory.engine:connect:131 - ============================================================
2025-11-07 12:37:45.689 | INFO     | data_processing.temporal_kg_engine.in_memory.engine:connect:132 - INITIALIZING IN-MEMORY GRAPH
2025-11-07 12:37:45.690 | INFO     | data_processing.temporal_kg_engine.in_memory.engine:connect:133 - ============================================================
2025-11-07 12:37:45.690 | INFO     | data_processing.temporal_kg_engine.base:download_data:124 - ============================================================
2025-11-07 12:37:45.691 | INFO     | data_processing.temporal_kg_engine.base:download_data:125 - Downloading Data from S3
2025-11-07 12:37:45.691 | INFO     | data_processing.temporal_kg_engine.base:download_data:126 - ============================================================
2025-11-07 12:37:45.692 | INFO     | data_processing.temporal_kg_engine.base:download_data:161 - =====

In [6]:
graph = engine.graph

In [18]:
import networkx as nx

In [22]:
graph = nx.MultiDiGraph()
graph.add_edge("B", "A", key="knows")

list(nx.all_shortest_paths(graph.to_undirected(), source="A", target="B"))

[['A', 'B']]

In [8]:
nodes = list(graph.nodes)
print(f"Total nodes in graph: {len(nodes)}")
source = nodes[0]
target = nodes[1]

paths = nx.all_shortest_paths(graph, source=source, target=target)

Total nodes in graph: 580434


In [16]:
source = "39ad38286d78e844fd795a54cfee63f3"
target = "6a1aea857162e3dd4d14839aee552066"
print(f"Finding paths from {source} to {target}")

paths = list(nx.all_shortest_paths(graph, source=source, target=target))

Finding paths from 39ad38286d78e844fd795a54cfee63f3 to 6a1aea857162e3dd4d14839aee552066


In [17]:
paths[0]

['39ad38286d78e844fd795a54cfee63f3', '6a1aea857162e3dd4d14839aee552066']

In [ ]:
engine.graph_backup_path

In [ ]:
nodes_pl = engine.nodes_df

In [ ]:
import polars as pl

node_name = "Ethanol"
node_name = "Abdomen, Acute"
node_name_list = ["Calcium", "Abdomen, Acute"]
node_data = nodes_pl.filter(pl.col("node_name").is_in(node_name_list))

for row in node_data.iter_rows(named=True):
    print(row["node_id"], row["node_name"])

In [ ]:
import polars as pl

node_id = "6239d7c38188ac8b2dbca6a63d87e919"
# node_id = "6a1aea857162e3dd4d14839aee552066"
node_data = nodes_pl.filter(pl.col("node_id") == node_id)

node_data["node_name"].item()